# AgriSense - Feature Engineering &amp; Yield Prediction

Welcome to Notebook 08! This is a massive milestone in the AgriSense project. Today we transition from **Data Preparation** strictly into **Machine Learning**.

### Goals for this Notebook:
**Part 1: The Final Data Features (1.3, 1.4, 1.6)**
*   Compute **Day-over-Day Price Percent Change** (the "+12%" you see on dashboard tiles).
*   Perform **Season Encoding** (turning Months into Kharif/Rabi/Zaid) because Machine Learning models require numbers, not strings.
*   **Save the Engineered Dataset** as our "Single Source of Truth".

**Part 2: Train the Yield Predictor Model (2.1)**
*   Train a **Random Forest Regressor** to predict `yield_quintal_per_acre`.
*   Evaluate the model using RMSE and R² Score.
*   Analyze Feature Importance.
*   Save the model so our FastAPI backend can use it for the frontend Yield Prediction page!

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# Machine Learning Libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

warnings.filterwarnings('ignore')

# Create directories for saving data and models
Path("../data/processed").mkdir(parents=True, exist_ok=True)
Path("../models").mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
print("Libraries imported successfully!")

## Part 1: Final Feature Engineering &amp; Saving Data

Let's load the enhanced dataset from our last notebook, add the final features, and establish our master dataset.

In [ ]:
# Load the enhanced dataset. 
# (If it doesn't exist, we will mock a quick version so this notebook runs independently)
try:
    df = pd.read_csv("../data/processed/crop_prices_enhanced.csv")
    df['date'] = pd.to_datetime(df['date'])
except FileNotFoundError:
    print("Previous dataset not found. Generating a mock dataset for continuity...")
    dates = pd.date_range("2024-01-01", periods=100)
    df = pd.DataFrame({
        'date': dates,
        'commodity': ['Wheat'] * 50 + ['Rice'] * 50,
        'modal_price': np.random.normal(2500, 200, 100)
    })
    df = df.sort_values(['commodity', 'date']).reset_index(drop=True)

# ---------------------------------------------------------
# 1.3 - Price Change Percentage (Day-over-Day)
# ---------------------------------------------------------
# The "+12%" number on your dashboard tiles.
df['price_pct_change'] = df.groupby('commodity')['modal_price'].pct_change() * 100

# Fill the first NaN value for each crop with 0 to prevent ML errors
df['price_pct_change'] = df['price_pct_change'].fillna(0)

# ---------------------------------------------------------
# 1.4 - Season Encoding
# ---------------------------------------------------------
# Extract month from date
df['month'] = df['date'].dt.month

# Map month to Indian Agricultural Seasons
def get_season(m):
    if m in [7, 8, 9, 10]: 
        return 'Kharif' # Monsoon crops
    elif m in [11, 12, 1, 2, 3]: 
        return 'Rabi'   # Winter crops
    else: 
        return 'Zaid'   # Summer crops

df['season_name'] = df['month'].apply(get_season)

# ML models need numbers, not strings! Let's encode the seasons.
# 0 = Kharif, 1 = Rabi, 2 = Zaid
season_mapping = {'Kharif': 0, 'Rabi': 1, 'Zaid': 2}
df['season_encoded'] = df['season_name'].map(season_mapping)

display(df[['date', 'commodity', 'modal_price', 'price_pct_change', 'season_name', 'season_encoded']].head())

In [ ]:
# ---------------------------------------------------------
# 1.6 - Save the Engineered Dataset
# ---------------------------------------------------------
output_path = Path('../data/processed/agrisense_features.csv')
df.to_csv(output_path, index=False)

# THIS IS CRITICAL FOR DATA PIPELINES:
print(f"✅ SUCCESS: Saved engineered dataset to {output_path}")
print(f"📊 Final Shape: {df.shape}")
print(f"📋 Final Columns: {list(df.columns)}")
print("\n" + "="*80)
print("🔒 This file is the single source of truth for all ML models and APIs. Never modify raw data.")
print("="*80)

## Part 2: Train Yield Predictor Model (2.1)

Now, let's build the predictive engine for the **Yield Prediction Page**.
Since our price data above doesn't have deep agricultural soil/fertilizer metrics, we will simulate a realistic agronomy dataset to train our Random Forest model.

### Why Random Forest and not Linear Regression?
Agriculture is highly **non-linear**. 
In Linear Regression, the formula implies: *"If rain increases, yield increases endlessly."*
In reality, crops need rain, but **too much rain causes floods and destroys the yield**! Random Forests use decision trees that perfectly capture these "Goldilocks zones" (e.g., Yield is good IF rain is &gt; 100mm AND rain is &lt; 300mm).

In [ ]:
# Generate realistic agricultural data for our Yield Model
np.random.seed(42)
n_samples = 2000

yield_df = pd.DataFrame({
    'crop_type': np.random.choice(['Wheat', 'Rice', 'Maize', 'Sugarcane'], n_samples),
    'soil_type': np.random.choice(['Alluvial', 'Black', 'Red', 'Clay'], n_samples),
    'season': np.random.choice(['Kharif', 'Rabi', 'Zaid'], n_samples),
    'rainfall': np.random.uniform(50, 500, n_samples),        # mm
    'fertilizer_amount': np.random.uniform(20, 150, n_samples) # kg/acre
})

# Let's simulate a non-linear target variable (Yield) based on the features
# Example: Rice loves rain and clay. Wheat prefers less rain. 
# Too much rain (&gt;400) hurts everything. Too much fertilizer (&gt;130) burns crops.
def calculate_mock_yield(row):
    base_yield = 15 # baseline quintals per acre
    
    # Fertilizer curve (Goldilocks zone around 80-100)
    fert_bonus = row['fertilizer_amount'] * 0.1 if row['fertilizer_amount'] &lt; 120 else -10
    
    # Rain curve (Goldilocks zone)
    if row['crop_type'] == 'Rice':
        rain_bonus = row['rainfall'] * 0.05 if row['rainfall'] &lt; 450 else -15
    else:
        rain_bonus = row['rainfall'] * 0.03 if row['rainfall'] &lt; 250 else -20
        
    return max(5, base_yield + fert_bonus + rain_bonus + np.random.normal(0, 3))

yield_df['yield_quintal_per_acre'] = yield_df.apply(calculate_mock_yield, axis=1)
display(yield_df.head())

### Preprocessing &amp; Train-Test Split
Before training, we must convert categorical strings (`crop_type`, `soil_type`, `season`) into numbers using `pd.get_dummies()`. This is called **One-Hot Encoding**.
Then, we split our data 80% for training (studying) and 20% for testing (exam).

In [ ]:
# 1. Feature Encoding (Turn text into 1s and 0s)
# drop_first=True prevents the "dummy variable trap" (multicollinearity)
X = pd.get_dummies(yield_df.drop('yield_quintal_per_acre', axis=1), drop_first=True)
y = yield_df['yield_quintal_per_acre']

# 2. Train/Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

### Train the Random Forest Regressor

In [ ]:
# Initialize the model with 100 decision trees
model = RandomForestRegressor(n_estimators=100, random_state=42)

# Train the model (This is where the math happens!)
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the Performance
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n" + "="*50)
print("🎯 MODEL PERFORMANCE METRICS")
print("="*50)
print(f"RMSE (Root Mean Square Error): {rmse:.2f} quintals/acre")
print(f"R² Score:                      {r2:.3f} ({(r2*100):.1f}% accuracy variance)")
print("="*50)
print("Interpretation:")
print(f"- The model's predictions are off by an average of {rmse:.2f} quintals.")
print(f"- The model successfully explains {(r2*100):.1f}% of the factors affecting crop yield.")

### Analyze Feature Importance
Let's ask the Random Forest which features were the most important when making decisions.

In [ ]:
importances = model.feature_importances_
feature_names = X.columns

# Build a dataframe to easily sort and plot
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(7), palette='viridis')
plt.title('Top 7 Most Important Features for Predicting Yield', fontsize=16)
plt.xlabel('Relative Importance')
plt.ylabel('Feature')
plt.show()

### Save the Model &amp; Test Prediction
We use `joblib` to save the trained model as a `.pkl` (pickle) file. The FastAPI backend will load this exact file when a user clicks "Predict Yield" on the frontend website!

In [ ]:
# Save the model
model_path = '../models/yield_predictor.pkl'
joblib.dump(model, model_path)
print(f"💾 Model successfully saved to: {model_path}")

# Get the exact column order the model expects
expected_columns = list(X.columns)

# ---------------------------------------------------------
# Test Prediction Example
# ---------------------------------------------------------
print("\n🔮 PREDICTION EXAMPLE")

# Create a sample input matching the exact structure of X_train
sample_input = pd.DataFrame(columns=expected_columns)
sample_input.loc[0] = 0 # Fill everything with 0 first

# Let's say a farmer inputs: 100mm rain, 85kg fertilizer, Wheat, Alluvial soil, Rabi season
sample_input['rainfall'] = 100.0
sample_input['fertilizer_amount'] = 85.0
sample_input['crop_type_Wheat'] = 1   # Set the dummy variable for Wheat to 1
sample_input['soil_type_Alluvial'] = 1
sample_input['season_Rabi'] = 1

prediction = model.predict(sample_input)[0]
print(f"Predicted Yield for sample input: {prediction:.2f} quintals per acre")

## 🎉 Congratulations!

### What we accomplished:
1. **Engineered the Final Dataset:** You added `price_pct_change` and `season_encoded` to prepare the data for APIs.
2. **Saved Master File:** `agrisense_features.csv` is now the golden source for the backend.
3. **Built AI!:** You successfully trained and exported a sophisticated **Random Forest Regressor** that understands the complex, non-linear relationships of agriculture.

### Next Steps:
The Data Science pipeline is complete! 
Next, we will move to the **Backend (FastAPI)**. We need to create an API route (e.g., `POST /predict/yield`) that loads our `yield_predictor.pkl` model, accepts JSON input from the React frontend, and returns the predicted yield!